# Complex CNN — Evaluation & Error Analysis

## 1. Load cấu hình, test data, checkpoint

In [18]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "src/complex_cnn.py").is_file():
    raise FileNotFoundError("Mở notebook từ thư mục gốc repo hoặc notebooks/.")
sys.path.insert(0, str(ROOT))

from src.complex_cnn import BATCH_SIZE, IMG_SIZE, zscore_normalize

MODEL_PATH = ROOT / "models/complex_cnn_best.keras"
TEST_DIR = ROOT / "data/test"
OUTPUT_DIR = ROOT / "outputs/complex_cnn"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not MODEL_PATH.is_file() or not TEST_DIR.is_dir():
    raise FileNotFoundError("Cần có models/complex_cnn_best.keras và data/test/.")

print("TensorFlow:", tf.__version__)
print("Model:", MODEL_PATH)
print("Test split:", TEST_DIR)

TensorFlow: 2.20.0
Model: /Volumes/Lexar E6/Code/Project1-DeepLearning-PlantVillage/models/complex_cnn_best.keras
Test split: /Volumes/Lexar E6/Code/Project1-DeepLearning-PlantVillage/data/test


In [ ]:
eval_datagen = ImageDataGenerator(preprocessing_function=zscore_normalize)
test_generator = eval_datagen.flow_from_directory(
    str(TEST_DIR),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False,
)

class_names = [
    name for name, index in sorted(
        test_generator.class_indices.items(), key=lambda item: item[1]
    )
]
assert len(class_names) == 38

best_model = tf.keras.models.load_model(MODEL_PATH)
print("Class count:", len(class_names))
print("Test images:", test_generator.samples)

Found 8162 images belonging to 38 classes.


## 2. Test loss, accuracy và macro metrics

In [ ]:
test_generator.reset()
test_loss, test_accuracy = best_model.evaluate(test_generator, verbose=1)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4%}")

In [ ]:
test_generator.reset()
y_prob = best_model.predict(test_generator, verbose=1)
y_pred = np.argmax(y_prob, axis=1)
y_true = test_generator.classes

assert len(y_true) == len(y_pred)
assert test_generator.shuffle is False

accuracy = accuracy_score(y_true, y_pred)
print(f"Accuracy: {accuracy:.4%}")
print(f"Macro Precision: {precision_score(y_true, y_pred, average='macro', zero_division=0):.4%}")
print(f"Macro Recall: {recall_score(y_true, y_pred, average='macro', zero_division=0):.4%}")
print(f"Macro F1: {f1_score(y_true, y_pred, average='macro', zero_division=0):.4%}")
print(f"Weighted F1: {f1_score(y_true, y_pred, average='weighted', zero_division=0):.4%}")

if not np.isclose(accuracy, test_accuracy, atol=1e-5):
    raise ValueError("evaluate() và predictions không khớp. Kiểm tra shuffle.")

np.savez_compressed(
    OUTPUT_DIR / "test_predictions.npz", y_true=y_true, y_pred=y_pred, y_prob=y_prob
)

## 3. Classification report và confusion matrix

In [ ]:
report = classification_report(
    y_true, y_pred, labels=range(len(class_names)),
    target_names=class_names, output_dict=True, zero_division=0
)
report_df = pd.DataFrame(report).T
report_df.to_csv(OUTPUT_DIR / "classification_report.csv")

class_report = report_df.loc[class_names]
display(class_report.sort_values("recall").head(10).round(4))

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=range(len(class_names)))
cm_normalized = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)

fig, ax = plt.subplots(figsize=(17, 15))
im = ax.imshow(cm_normalized, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(class_names)))
ax.set_yticks(range(len(class_names)))
ax.set_xticklabels(class_names, rotation=90, fontsize=7)
ax.set_yticklabels(class_names, fontsize=7)
ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")
ax.set_title("Complex CNN - Normalized Confusion Matrix")
fig.colorbar(im, ax=ax)
fig.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=200, bbox_inches="tight")
plt.show()

## 4. Error analysis

In [ ]:
target_classes = [
    "Tomato___Early_blight",
    "Tomato___Target_Spot",
    "Potato___healthy",
]

rows = []
for true_class in target_classes:
    true_idx = test_generator.class_indices[true_class]
    for pred_idx, count in enumerate(cm[true_idx]):
        if pred_idx != true_idx and count > 0:
            rows.append({
                "True Class": true_class,
                "Predicted Class": class_names[pred_idx],
                "Wrong Predictions": int(count),
            })

errors_df = pd.DataFrame(rows).sort_values("Wrong Predictions", ascending=False)
errors_df.to_csv(OUTPUT_DIR / "top_confusions.csv", index=False)
display(errors_df)

In [ ]:
true_class = "Tomato___Early_blight"
pred_class = "Tomato___Septoria_leaf_spot"
true_idx = test_generator.class_indices[true_class]
pred_idx = test_generator.class_indices[pred_class]
indices = np.where((y_true == true_idx) & (y_pred == pred_idx))[0][:12]

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for ax, idx in zip(axes.flat, indices):
    image = Image.open(test_generator.filepaths[idx]).convert("RGB")
    ax.imshow(image)
    ax.set_title(f"Confidence: {y_prob[idx, pred_idx]:.1%}")
    ax.axis("off")
for ax in axes.flat[len(indices):]:
    ax.axis("off")

fig.suptitle("Early Blight Misclassified as Septoria Leaf Spot")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "error_examples.png", dpi=200, bbox_inches="tight")
plt.show()

## 5. Training curves từ history đã lưu

In [ ]:
history_path = OUTPUT_DIR / "complex_cnn_history.csv"
if not history_path.exists():
    raise FileNotFoundError(f"Missing history file: {history_path}")

history_df = pd.read_csv(history_path)
epochs = range(1, len(history_df) + 1)

for metric, ylabel in (("accuracy", "Accuracy"), ("loss", "Loss")):
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(epochs, history_df[metric], label=f"Train {ylabel}")
    ax.plot(epochs, history_df[f"val_{metric}"], label=f"Validation {ylabel}")
    ax.set(xlabel="Epoch", ylabel=ylabel, title=f"Complex CNN - {ylabel}")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.savefig(OUTPUT_DIR / f"{metric}_curve.png", dpi=200, bbox_inches="tight")
    plt.show()

best_idx = history_df["val_loss"].idxmin()
print("Best epoch:", best_idx + 1)
print("Best val_loss:", round(history_df.loc[best_idx, "val_loss"], 4))